# ForkWise Demo Notebook

This notebook is the guided demo runner for the **current integrated ML system**.

It is designed to help you prove, in order:

1. the deployed services are healthy
2. production traffic can reach the serving system
3. inference works
4. feedback is captured
5. feedback is turned into retraining data
6. retraining is triggered
7. evaluation happens before publishing
8. serving picks up the updated model
9. Prometheus and Grafana are deployed for live monitoring
10. the safeguarding story can be pointed to in real code

## Scope

This notebook focuses on the internal ML system only:

- `data/`
- `training/`
- `serving/`
- `infra/` monitoring and deployment surfaces

It does **not** depend on Mealie for the core demo.

## MLflow / Prometheus / Grafana guidance

- **MLflow is optional for this notebook.** Do not force MLflow into the demo unless it is already live and populated.
- **Prometheus and Grafana should be shown** for serving, data, and platform monitoring.
- **Training should be shown through training logs and evaluation output**, not as a long-running Grafana metrics endpoint.

## Before you run this notebook

Open these in separate terminals and keep them running:

Terminal A:

```bash
ssh -N -L 8000:127.0.0.1:30080 -L 9000:127.0.0.1:30090 cc@129.114.27.72
```

Terminal B:

```bash
ssh -L 8001:127.0.0.1:8001 cc@129.114.27.72 "kubectl port-forward svc/subst-feedback -n forkwise-data 8001:8001"
```


## 1. Imports and helper functions

This cell only sets up the notebook helpers.

What this section demonstrates:
- nothing user-facing yet
- just reusable helpers so the rest of the demo is clean and repeatable

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path
from urllib import request as urllib_request
from urllib.error import HTTPError, URLError


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "serving" / "sample_data" / "input_sample.json").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")


REPO_ROOT = find_repo_root()
NODE = os.environ.get("NODE", "cc@129.114.27.72")
SERVING_URL = os.environ.get("SERVING_URL", "http://localhost:8000")
FEEDBACK_URL = os.environ.get("FEEDBACK_URL", "http://localhost:8001")
MONITORING_NAMESPACE = os.environ.get("MONITORING_NAMESPACE", "monitoring-proj01")
SAMPLE_INPUT = REPO_ROOT / "serving" / "sample_data" / "input_sample.json"


def run(cmd, timeout=180, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def kssh(cmd, timeout=180, check=True):
    escaped = cmd.replace('"', '\\"')
    return run(f'ssh {NODE} "{escaped}"', timeout=timeout, check=check)


def http_json(method, url, payload=None, timeout=20):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib_request.Request(url, data=data, method=method)
    req.add_header("Content-Type", "application/json")
    with urllib_request.urlopen(req, timeout=timeout) as resp:
        body = resp.read().decode("utf-8")
        return resp.status, json.loads(body)


def http_text(method, url, payload=None, timeout=20):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib_request.Request(url, data=data, method=method)
    if payload is not None:
        req.add_header("Content-Type", "application/json")
    with urllib_request.urlopen(req, timeout=timeout) as resp:
        return resp.status, resp.read().decode("utf-8")


def pretty(obj):
    print(json.dumps(obj, indent=2))


print("Repo root:", REPO_ROOT)
print("Node:", NODE)
print("Serving URL:", SERVING_URL)
print("Feedback URL:", FEEDBACK_URL)


## 2. Preflight: confirm the deployed services are present

What this proves:
- the serving namespace exists
- the data namespace exists
- the demo-critical workloads are visible to Kubernetes
- the monitoring namespace exists if Prometheus/Grafana were deployed

What to say:
- "This is the deployed system on Chameleon."
- "I am checking the live pods, jobs, and CronJobs before I run the internal loop."

Needs Prometheus/Grafana?
- Not yet, but this step should confirm whether the monitoring namespace exists

Needs MLflow?
- No


In [ ]:
kssh("kubectl get pods -n forkwise-serving", timeout=60)
kssh("kubectl get pods -n forkwise-data", timeout=60)
kssh("kubectl get cronjobs -n forkwise-data", timeout=60)
kssh(f"kubectl get pods -n {MONITORING_NAMESPACE}", timeout=60, check=False)


## 3. Service health: serving and feedback

What this proves:
- serving is reachable on the expected local tunnel
- feedback is reachable on the expected local tunnel

What to say:
- "The serving API and feedback API are both live."
- "The health endpoint also shows the loaded model version and serving version."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
status, serving_health = http_json("GET", f"{SERVING_URL}/health")
print("Serving status:", status)
pretty(serving_health)

status, feedback_health = http_json("GET", f"{FEEDBACK_URL}/health")
print("Feedback status:", status)
pretty(feedback_health)


## 4. Emulated production traffic is active

What this proves:
- the production-data generator is running
- traffic is being replayed into the system

What to say:
- "This generator is our stand-in for real production traffic."
- "It hits service endpoints and continuously produces data for monitoring and retraining."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
kssh("kubectl logs deployment/data-generator -n forkwise-data --tail=30", timeout=60)


## 5. Run one live prediction

What this proves:
- the inference endpoint works
- the model responds with substitutions
- `request_id`, `model_version`, and `serving_version` are returned and can be preserved by the product

What to say:
- "This is the API contract the product uses for inference."
- "The response includes the request ID and model version so feedback can be linked back to the exact prediction."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
with open(SAMPLE_INPUT, "r", encoding="utf-8") as f:
    predict_payload = json.load(f)

status, predict_json = http_json("POST", f"{SERVING_URL}/predict", predict_payload)
print("Predict status:", status)
pretty(predict_json)

REQUEST_ID = predict_json["request_id"]
MODEL_VERSION_BEFORE = predict_json["model_version"]

print("REQUEST_ID =", REQUEST_ID)
print("MODEL_VERSION_BEFORE =", MODEL_VERSION_BEFORE)


## 6. Capture feedback for that exact prediction

What this proves:
- feedback is captured as part of the live loop
- the same `request_id` is preserved
- feedback can be tied to the exact model version that produced the prediction

What to say:
- "This closes the user-feedback loop."
- "The same request ID is preserved from inference to feedback."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
feedback_payload = {
    "request_id": REQUEST_ID,
    "recipe_id": str(predict_json["recipe_id"]),
    "missing_ingredient": predict_json["missing_ingredient"],
    "suggested_substitution": predict_json["substitutions"][0]["ingredient"],
    "user_accepted": True,
    "model_version": MODEL_VERSION_BEFORE,
}

status, feedback_json = http_json("POST", f"{FEEDBACK_URL}/feedback", feedback_payload)
print("Feedback status:", status)
pretty(feedback_json)

kssh("kubectl logs deployment/subst-feedback -n forkwise-data --tail=25", timeout=60)


## 7. Lower the retraining threshold for the demo

What this does:
- makes one new feedback sample enough to trigger the retraining path in the demo

What to say:
- "For demo purposes, I lower the threshold so one new feedback event is enough."
- "In normal operation, this threshold should be higher."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
kssh(
    "kubectl patch configmap forkwise-data-config -n forkwise-data "
    "--type merge "
    "-p '{\"data\":{\"MIN_NEW_SAMPLES\":\"1\"}}'",
    timeout=60,
)


## 8. Run the batch pipeline once

What this proves:
- saved feedback is incorporated into new training data
- the batch step writes a processed dataset
- the batch step writes a retraining trigger

What to say:
- "The batch pipeline reads the saved feedback logs."
- "It validates and compiles a retraining dataset."
- "Then it emits a retraining trigger into object storage."

Needs Prometheus/Grafana?
- Not required in this step

Needs MLflow?
- No


In [ ]:
kssh("kubectl delete job batch-smoke -n forkwise-data --ignore-not-found=true", check=False, timeout=60)
kssh("kubectl create job batch-smoke --from=cronjob/batch-pipeline -n forkwise-data", timeout=60)
kssh("kubectl wait --for=condition=complete job/batch-smoke -n forkwise-data --timeout=240s", timeout=260)
kssh("kubectl logs -n forkwise-data job/batch-smoke --tail=220", timeout=60)


## 9. Run the training job once

What this proves:
- the training trigger is consumed
- retraining runs live in the deployment
- evaluation happens before publishing
- the model is only published if it passes the quality gate

What to say:
- "This is the live retraining stage."
- "The training code evaluates the model before publishing it."
- "This is the training-role evaluation requirement."

Needs Prometheus/Grafana?
- No
- Training is better shown through logs and evaluation output

Needs MLflow?
- No for the minimal demo
- Only show MLflow separately if it already exists and has real runs


In [ ]:
kssh("kubectl delete job training-smoke -n forkwise-data --ignore-not-found=true", check=False, timeout=60)
kssh("kubectl create job training-smoke --from=cronjob/training-trigger -n forkwise-data", timeout=60)
kssh("kubectl wait --for=condition=complete job/training-smoke -n forkwise-data --timeout=360s", timeout=380)
kssh("kubectl logs -n forkwise-data job/training-smoke --tail=260", timeout=60)


## 10. Verify serving picked up the updated model

What this proves:
- the updated model artifact was published
- serving reloaded the new model
- the full path from feedback to updated serving is working end-to-end

What to say:
- "The changed model version proves the retraining-and-reload loop completed."

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
MODEL_VERSION_AFTER = None

for attempt in range(12):
    status, serving_health_after = http_json("GET", f"{SERVING_URL}/health")
    print(f"Attempt {attempt + 1}: health status =", status)
    pretty(serving_health_after)
    MODEL_VERSION_AFTER = serving_health_after.get("model_version")
    if MODEL_VERSION_AFTER != MODEL_VERSION_BEFORE:
        print("Model version changed.")
        break
    print("Model version has not changed yet; waiting 10 seconds...")
    time.sleep(10)

print("MODEL_VERSION_BEFORE =", MODEL_VERSION_BEFORE)
print("MODEL_VERSION_AFTER =", MODEL_VERSION_AFTER)

status, predict_after = http_json("POST", f"{SERVING_URL}/predict", predict_payload)
print("Predict status after retraining:", status)
pretty(predict_after)


## 11. Show live metrics from serving

What this proves:
- serving exports Prometheus metrics
- Prometheus has something meaningful to scrape

What to say:
- "Prometheus scrapes this metrics endpoint."
- "Grafana visualizes these metrics in dashboards."

Needs Prometheus/Grafana?
- Yes, this is the right place to connect the live endpoint to Prometheus/Grafana

Needs MLflow?
- No


In [ ]:
status, metrics_text = http_text("GET", f"{SERVING_URL}/metrics")
print("Metrics status:", status)
interesting_lines = [line for line in metrics_text.splitlines() if line.startswith("subst_")]
print("\n".join(interesting_lines[:120]))


## 12. Show data monitoring and the monitoring stack

What this proves:
- the data side has live monitoring for drift / OOV / low-confidence style signals
- the monitoring namespace is present for Prometheus and Grafana

What to say:
- "Serving and data are the strongest live monitoring surfaces in the current system."
- "Training monitoring is shown through evaluation logs instead of a persistent metrics endpoint."

Needs Prometheus/Grafana?
- Yes

Needs MLflow?
- No


In [ ]:
kssh("kubectl delete job drift-smoke -n forkwise-data --ignore-not-found=true", check=False, timeout=60)
kssh("kubectl create job drift-smoke --from=cronjob/drift-monitor -n forkwise-data", timeout=60)
kssh("kubectl wait --for=condition=complete job/drift-smoke -n forkwise-data --timeout=240s", timeout=260)
kssh("kubectl logs -n forkwise-data job/drift-smoke --tail=220", timeout=60)
kssh(f"kubectl get pods -n {MONITORING_NAMESPACE}", timeout=60, check=False)
kssh(f"kubectl get svc -n {MONITORING_NAMESPACE}", timeout=60, check=False)


## 13. Optional: switch to Grafana now

If Grafana is exposed and working, this is the right point in the demo to switch to the Grafana UI.

What to show in Grafana:
- serving latency
- serving error rate
- serving traffic
- model loaded state
- feedback write errors
- any available platform health panels

Important:
- Grafana should be used to show **serving**, **data**, and **platform** monitoring
- do **not** use Grafana as the main proof of training evaluation
- training evaluation should remain the training-job logs and the quality gate output

MLflow note:
- only switch to MLflow separately if it is already live and your team wants to show historical experiment tracking
- it is not required for the core minimal demo


## 14. Safeguarding and code references to mention during the video

This notebook demonstrates live behavior. For the safeguarding section, open the following files in the editor and narrate the mechanisms:

- Privacy and accountability
  - `data/feedback_endpoint.py`
  - `serving/fastapi_onnx/serve_onnx.py`

- Transparency and robustness
  - `serving/fastapi_onnx/serve_onnx.py`
  - `serving/scripts/check_rollback.py`

- Fairness and evaluation
  - `training/evaluate.py`

- Data quality and drift
  - `data/batch_pipeline.py`
  - `data/drift_monitor.py`

- Prometheus / Grafana monitoring
  - `infra/k8s/platform/prometheus-configmap.yaml`
  - `infra/k8s/platform/grafana-dashboard-configmap.yaml`

Short narration:
- privacy: feedback logging avoids user identity fields
- transparency: the serving response includes `model_version` and `serving_version`
- accountability: request logs, feedback logs, and rollback records exist
- robustness: evaluation gates, model reload, and rollback exist
- fairness: per-cuisine metrics are computed during evaluation


## 15. Optional: Mealie and rollout-stack notes

Do not force these into the notebook unless they are already validated live.

- **Mealie** should be shown in a browser tab, not from this notebook
- **Canary / staging / production rollout** should only be shown if your live environment is already using that path successfully

The cleanest demo is:
1. run this notebook for the internal ML loop
2. switch to Grafana for monitoring
3. switch to Mealie in a browser only if the feature already works live


## 16. Cleanup: restore the normal feedback threshold

What this does:
- resets the demo-only lower threshold back to its normal value

Needs Prometheus/Grafana?
- No

Needs MLflow?
- No


In [ ]:
kssh(
    "kubectl patch configmap forkwise-data-config -n forkwise-data "
    "--type merge "
    "-p '{\"data\":{\"MIN_NEW_SAMPLES\":\"50\"}}'",
    timeout=60,
)
print("Restored MIN_NEW_SAMPLES to 50.")
